# 1. Import & Config

In [1]:
import sys, os, json
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import optuna
import shutil
import math

import pickle
import plotly.io as pio
import matplotlib.pyplot as plt

from pathlib import Path
from dataclasses import asdict

from config import (
    ExperimentConfig,
    TrainingConfig,
    OptunaConfig,
    DDPMTransformerConfig,
    FMConfig,
    AdamConfig,
    AdamWConfig,
    CosineSchedulerConfig,
    WarmupCosineSchedulerConfig,
    PlateauSchedulerConfig,
    CriterionConfig,
    BaseModelConfig,
    DataConfig,
)
from engine import Engine
from models import Diffusion, DiffusionTransformer, FlowMatchingModel
from utils import (
    setup_logging,
    setup_random_seed,
    ExperimentManager,
    make_all_dataloaders,
    load_variant
)
from utils.paths import DATASETS_DIR, PROCESSED_DIR, CHECKPOINTS_DIR, EXPERIMENTS_DIR

optuna.logging.set_verbosity(optuna.logging.WARNING)
logger = setup_logging()
logger.info("✓ Imports OK")

2026-06-07 04:17:05,490 - utils.setup - INFO - Logger is set up.
2026-06-07 04:17:05,492 - utils.setup - INFO - ✓ Imports OK


In [ ]:
exp_cfg = ExperimentConfig(
    name        = "v1_trial04_optimized",
    description = "baseline 16 assets — DDPM Transformer",
    random_seed = 72,
    device      = "cuda:0",

    skip_optuna   = True,
    skip_training = False,

    data = DataConfig(
        seq_dims = ("W", "A"),
        feat_dims = ("D", "C")
    ),

    model = DDPMTransformerConfig(
        d_model            = 1024,
        ff_mult            = 4,
        num_layers         = 3,
        num_attention_heads = 4,
        dropout            = 0.1,
        timesteps          = 1000,
    ),

    training = TrainingConfig(
        num_epochs         = 1000,
        max_grad_norm      = 1.0,
        save_checkpoint_freq = 10,
        # optimizer  = AdamWConfig(lr=1e-4, weight_decay=1e-2),
        # optimizer  = AdamWConfig(lr=1e-4, weight_decay=3.12e-07),
        optimizer  = AdamConfig(lr=0.00026034551721542293, weight_decay=6.261497338150341e-07),
        scheduler  = WarmupCosineSchedulerConfig(warmup_epochs=20, eta_min=1e-6),
        # scheduler   = PlateauSchedulerConfig( patience = 10, factor = 0.5, eta_min = 1e-6 ),
        criterion  = CriterionConfig("MSELoss"),
    ),
    optuna = OptunaConfig(
        n_trials         = 100,
        epochs_per_trial = 50,
        min_resource     = 5,
        max_resource     = 30,
        reduction_factor = 3,
        suggest_d_model  = [256, 512, 1024],
        suggest_ff_mult = [2, 4],
        suggest_n_heads = [4, 8, 16],
        suggest_n_layers = [2, 4, 8, 16, 32],
        suggest_dropout = [0.1, 0.2, 0.3],
        suggest_lr = [1e-4, 5e-4, 1e-3],
        suggest_weight_decay = [1e-7, 1e-6, 1e-5, 1e-4]
    ),
)

FEATURE_STORE_NAME    = "v1_trial04_w10_d20_asset16"
FEATURE_STORE_VARIANT = "robust_standard"

setup_random_seed(exp_cfg.random_seed)
DEVICE = torch.device(exp_cfg.device)
STORE_DIR = PROCESSED_DIR

## UI

In [3]:

m = exp_cfg.model
t = exp_cfg.training

print("=" * 60)
print("  V3-2 MODELING")
print("=" * 60)
print(f"  Experiment    : {exp_cfg.name}")
print(f"  Description   : {exp_cfg.description}")
print()
print(f"  Feature store : {FEATURE_STORE_NAME}  [{FEATURE_STORE_VARIANT}]")
print(f"  Store path    : {STORE_DIR}")
print()
print(f"  Model         : {m.name}  (d={m.d_model}, L={m.num_layers}, H={m.num_attention_heads})")
print(f"  Optimizer     : {t.optimizer.name}  (lr={t.optimizer.lr})")
print(f"  Scheduler     : {t.scheduler.name}")
print(f"  Criterion     : {t.criterion.name}")
print(f"  Epochs        : {t.num_epochs}")
print()
print(f"  Device        : {DEVICE}")
print(f"  Random seed   : {exp_cfg.random_seed}")
print(f"  Skip Optuna   : {exp_cfg.skip_optuna}")
print(f"  Skip Training : {exp_cfg.skip_training}")
print("=" * 60)

  V3-2 MODELING
  Experiment    : v1_trial04_optimized
  Description   : baseline 16 assets — DDPM Transformer

  Feature store : v1_trial04_w40_d20_asset16  [robust_standard]
  Store path    : /home/narodom.y@FUSION.LAB/research/01_processed

  Model         : ddpm_transformer  (d=1024, L=3, H=4)
  Optimizer     : adam  (lr=0.00026034551721542293)
  Scheduler     : warmup_cosine
  Criterion     : MSELoss
  Epochs        : 1000

  Device        : cuda:0
  Random seed   : 72
  Skip Optuna   : True
  Skip Training : False


# 2. Load Feature Store

In [4]:
data, scalers = load_variant(STORE_DIR, FEATURE_STORE_NAME, FEATURE_STORE_VARIANT)
loaders       = make_all_dataloaders(STORE_DIR, FEATURE_STORE_NAME, FEATURE_STORE_VARIANT, batch_sizes=exp_cfg.training.batch_sizes)

with open(STORE_DIR / f"{FEATURE_STORE_NAME}_{FEATURE_STORE_VARIANT}" / "meta.json") as f:
    meta = json.load(f)

SYMBOLS    = meta["cfg"]["symbols"]
x_scaler   = scalers["x"]
cond_scaler = scalers["cond"]

print("✓ Feature store loaded")

✓ Feature store loaded


## 2.1 Shape summary 

In [5]:
splits = ["train", "val", "test"]
print(f"\n{'split':<8} {'x':<30} {'cond':<25} {'n_windows'}")
print("-" * 75)
for s in splits:
    x    = data[s]["x"]
    cond = data[s]["cond"]
    print(f"{s:<8} {str(x.shape):<30} {str(cond.shape):<25} {x.shape[0]}")


split    x                              cond                      n_windows
---------------------------------------------------------------------------
train    (1302, 16, 40, 20, 4)          (1302, 16, 40, 11)        1302
val      (895, 16, 40, 20, 4)           (895, 16, 40, 11)         895
test     (890, 16, 40, 20, 4)           (890, 16, 40, 11)         890


## 2.2 Scaled check

In [6]:
x_train = data["train"]["x"]                          # (T, A, W, D, C)
x_flat  = x_train.reshape(-1, x_train.shape[-1])

channels = ["logdiff_close", "logdiff_high", "logdiff_low", "logdiff_volume"]
print(f"\n{'channel':<25} {'mean':>10} {'std':>10} {'min':>10} {'max':>10}")
print("-" * 65)
for i, ch in enumerate(channels):
    col = x_flat[:, i]
    print(f"{ch:<25} {col.mean():>10.4f} {col.std():>10.4f} {col.min():>10.4f} {col.max():>10.4f}")


channel                         mean        std        min        max
-----------------------------------------------------------------
logdiff_close                 0.0683     1.6069   -18.8869    31.4357
logdiff_high                  0.0795     1.6898   -17.0379    38.4848
logdiff_low                   0.0736     1.6821   -30.9613    29.6857
logdiff_volume               -0.0008     1.1168    -6.0605     8.6496


## 2.3 Inverse scale → log returns

In [7]:
sample_scaled = data["test"]["x"][0]        # (A, W, D, C)
shape         = sample_scaled.shape

x_inv = x_scaler.inverse_transform(
    sample_scaled.reshape(-1, shape[-1])    # (A*W*D, C)
).reshape(shape)                            # (A, W, D, C)

## 2.4 Reconstruct close price

In [8]:
init_price   = data["test"]["init_price"][0]     # (A,)
logret_close = x_inv[:, :, 0, 0]                 # (A, W) — D=0, Close channel
log_price    = np.cumsum(logret_close, axis=-1)
close_price  = init_price[:, None] * np.exp(log_price)

print(f"\n{'symbol':<8} {'init_price':>12} {'reconstructed[-1]':>20}")
print("-" * 45)
for i, sym in enumerate(SYMBOLS):
    print(f"{sym:<8} {init_price[i]:>12.2f} {close_price[i, -1]:>20.2f}")


symbol     init_price    reconstructed[-1]
---------------------------------------------
AAPL           161.80               130.32
AMZN           151.71               101.59
CAT            211.60               191.14
EEM             40.12                36.93
GOOGL          125.69               105.06
JNJ            160.19               152.62
JPM            113.98               101.20
KO              57.75                53.46
META           208.54               162.00
MSFT           270.12               235.97
NVDA            21.22                15.43
SPY            414.27               353.64
TLT            103.75                94.28
TSLA           328.33               216.65
GLD            184.04               168.05
XOM             76.02                83.34


## 2.5  Inverse sanity check

In [9]:
ohlcv_sample = data["test"]["ohlcv_raw"][0]   # (A, W, 4) — Close/High/Low/Volume
raw_close     = ohlcv_sample[:, :, 0]         # (A, W)

max_err  = np.abs(close_price - raw_close).max()
mean_err = np.abs(close_price - raw_close).mean()

print(f"{'symbol':<8} {'init_price':>12} {'recon[-1]':>12} {'raw[-1]':>12} {'max_err':>10}")
print("-" * 60)
for i, sym in enumerate(SYMBOLS):
    err = np.abs(close_price[i] - raw_close[i]).max()
    print(f"{sym:<8} {init_price[i]:>12.2f} {close_price[i,-1]:>12.2f} {raw_close[i,-1]:>12.2f} {err:>10.6f}")

print(f"\n  global max  abs err : {max_err:.2e}   {'✓ OK' if max_err < 1e-3 else '✗ CHECK'}")
print(f"  global mean abs err : {mean_err:.2e}")

symbol     init_price    recon[-1]      raw[-1]    max_err
------------------------------------------------------------
AAPL           161.80       130.32       130.15   0.218384
AMZN           151.71       101.59       102.31   1.116135
CAT            211.60       191.14       192.91   2.003845
EEM             40.12        36.93        36.81   0.135368
GOOGL          125.69       105.06       105.84   0.955917
JNJ            160.19       152.62       150.72   2.088318
JPM            113.98       101.20       103.08   2.176773
KO              57.75        53.46        52.99   0.529259
META           208.54       162.00       162.46   0.620544
MSFT           270.12       235.97       236.55   0.688538
NVDA            21.22        15.43        15.81   0.534037
SPY            414.27       353.64       353.78   0.173126
TLT            103.75        94.28        93.81   0.525208
TSLA           328.33       216.65       220.89   6.582764
GLD            184.04       168.05       168.57   0.56

## 2.6 DataLoader batch shape check

In [10]:
x_batch, cond_batch, init_batch, ohlcv_batch, date_batch, init_date_batch = next(iter(loaders["train"]))

print(f"  x          : {str(tuple(x_batch.shape)):<30} dtype={x_batch.dtype}")
print(f"  cond       : {str(tuple(cond_batch.shape)):<30} dtype={cond_batch.dtype}")
print(f"  init_price : {str(tuple(init_batch.shape)):<30} dtype={init_batch.dtype}")
print(f"  ohlcv_raw  : {str(tuple(ohlcv_batch.shape)):<30} dtype={ohlcv_batch.dtype}")
print(f"  dates      : {str(tuple(date_batch.shape)):<30} dtype={date_batch.dtype}")
print(f"  init_date  : {str(tuple(init_date_batch.shape)):<30} dtype={init_date_batch.dtype}")

for name, t in [("x", x_batch), ("cond", cond_batch), ("init_price", init_batch), ("ohlcv_raw", ohlcv_batch), ("dates", date_batch), ("init_date", init_date_batch)]:
    has_nan = torch.isnan(t).any().item()
    has_inf = torch.isinf(t).any().item()
    status  = "✓ clean" if not has_nan and not has_inf else f"✗  nan={has_nan}  inf={has_inf}"
    print(f"  {name:<10} : {status}")

  x          : (64, 16, 40, 20, 4)            dtype=torch.float32
  cond       : (64, 16, 40, 11)               dtype=torch.float32
  init_price : (64, 16)                       dtype=torch.float32
  ohlcv_raw  : (64, 16, 40, 4)                dtype=torch.float32
  dates      : (64, 40)                       dtype=torch.int64
  init_date  : (64,)                          dtype=torch.int64
  x          : ✓ clean
  cond       : ✓ clean
  init_price : ✓ clean
  ohlcv_raw  : ✓ clean
  dates      : ✓ clean
  init_date  : ✓ clean


# 3.

# 3.1 UI

In [11]:
def make_exp_name(exp_cfg, window_size: int, sequence_depth: int) -> str:
    m = exp_cfg.model
    s = exp_cfg.training.scheduler
    name_tag      = exp_cfg.name.replace(" ", "_")
    model_tag     = m.name.replace("_transformer", "").replace("_matching", "")
    scheduler_tag = s.name.replace("_", "-")
    seq_dims_tag  = "".join(exp_cfg.data.seq_dims)
    feat_dims_tag = "".join(exp_cfg.data.feat_dims)
    seq_feat_tag  = f"seq{seq_dims_tag}_feat{feat_dims_tag}"
    return f"{name_tag}_{model_tag}_{scheduler_tag}_w{window_size}_d{sequence_depth}_{seq_feat_tag}_{FEATURE_STORE_VARIANT}"

In [12]:
# def make_exp_name(exp_cfg, window_size: int, sequence_depth: int) -> str:
#     m = exp_cfg.model
#     s = exp_cfg.training.scheduler
#     name_tag      = exp_cfg.name.replace(" ", "_")
#     model_tag     = m.name.replace("_transformer", "").replace("_matching", "")
#     scheduler_tag = s.name.replace("_", "-")

#     feat_dims_tag = "".join(exp_cfg.data.feat_dims)
#     seq_feat_tag  = f"seq{seq_dims_tag}_feat{feat_dims_tag}"

#     return f"{name_tag}_{model_tag}_{scheduler_tag}_w{window_size}_d{sequence_depth}_{seq_feat_tag}"

In [13]:
EXP_NAME = make_exp_name(exp_cfg, meta["cfg"]["window_size"], meta["cfg"]["sequence_depth"])
EXP_DIR  = EXPERIMENTS_DIR / EXP_NAME

# สร้าง subdirs
(EXP_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(EXP_DIR / "optuna").mkdir(parents=True, exist_ok=True)
(EXP_DIR / "plots").mkdir(parents=True, exist_ok=True)

print(f"  EXP_NAME : {EXP_NAME}")
print(f"  EXP_DIR  : {EXP_DIR}")

  EXP_NAME : v1_trial04_optimized_ddpm_warmup-cosine_w40_d20_seqWA_featDC_robust_standard
  EXP_DIR  : /home/narodom.y@FUSION.LAB/research/02_experiments/v1_trial04_optimized_ddpm_warmup-cosine_w40_d20_seqWA_featDC_robust_standard


In [14]:
# experiment_config.json
snap = {
    "exp_name"  : EXP_NAME,
    "exp_cfg"   : asdict(exp_cfg),
    "feature_store": {
        "name"   : FEATURE_STORE_NAME,
        "variant": FEATURE_STORE_VARIANT,
        "path"   : str(STORE_DIR),
    },
}
with open(EXP_DIR / "experiment_config.json", "w") as f:
    json.dump(snap, f, indent=2, default=str)

# meta.json — copy จาก feature store
shutil.copy(
    STORE_DIR / f"{FEATURE_STORE_NAME}_{FEATURE_STORE_VARIANT}" / "meta.json",
    EXP_DIR / "meta.json",
)

print(f"  ✓ experiment_config.json saved")
print(f"  ✓ meta.json copied")

  ✓ experiment_config.json saved
  ✓ meta.json copied


## 3.2 model registry

In [15]:
def build_model(exp_cfg, meta: dict, data: dict) -> nn.Module:
    m = exp_cfg.model

    B, A, W, D, C = data["train"]["x"].shape
    _, _, _, F    = data["train"]["cond"].shape

    x_dim_map    = {"A": A, "W": W, "D": D, "C": C}
    cond_dim_map = {"A": A, "W": W, "F": F}

    # seq ของ cond = intersection ของ seq_dims กับ dims ที่ cond มี
    cond_seq_dims  = [d for d in exp_cfg.data.seq_dims  if d in cond_dim_map]
    cond_feat_dims = [d for d in exp_cfg.data.feat_dims if d in cond_dim_map]
    if "F" not in cond_feat_dims:
        cond_feat_dims = cond_feat_dims + ["F"]

    input_dim = math.prod(x_dim_map[d] for d in exp_cfg.data.feat_dims)
    cond_dim  = math.prod(cond_dim_map[d] for d in cond_feat_dims)

    print(f"  seq_dims   : {exp_cfg.data.seq_dims}")
    print(f"  feat_dims  : {exp_cfg.data.feat_dims}")
    print(f"  input_dim  : {input_dim}  ({' * '.join(str(x_dim_map[d]) for d in exp_cfg.data.feat_dims)})")
    print(f"  cond_dim   : {cond_dim}")

    if isinstance(m, DDPMTransformerConfig):
        backbone = DiffusionTransformer(
            input_dim           = input_dim,
            cond_dim            = cond_dim,
            d_model             = m.d_model,
            num_layers          = m.num_layers,
            num_attention_heads = m.num_attention_heads,
            dim_feedforward     = m.dim_feedforward,
            dropout             = m.dropout,
        )
        return Diffusion(
            model      = backbone,
            timesteps  = m.timesteps,
            beta_start = m.beta_start,
            beta_end   = m.beta_end,
        ).to(DEVICE)

    elif isinstance(m, FMConfig):
        return FlowMatchingModel(cfg=m).to(DEVICE)

    raise ValueError(f"Unknown model config: {type(m)}")

model = build_model(exp_cfg, meta, data)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Model     : {exp_cfg.model.name}")
print(f"  Params    : {n_params:,}")

  seq_dims   : ('W', 'A')
  feat_dims  : ('D', 'C')
  input_dim  : 80  (20 * 4)
  cond_dim   : 11
  Model     : ddpm_transformer
  Params    : 53,715,024


# 4. Optuna 

# 4.1 Objective function

In [16]:
def objective(trial: optuna.Trial) -> float:

    o = exp_cfg.optuna
    m = exp_cfg.model
    t = exp_cfg.training

    # ── Search space ──────────────────────────────────────────
    d_model = trial.suggest_categorical("d_model",  o.suggest_d_model)
    ff_mult = trial.suggest_categorical("ff_mult",  o.suggest_ff_mult)
    n_heads = trial.suggest_categorical("n_heads",  o.suggest_n_heads)
    n_layers= trial.suggest_int(        "n_layers", o.suggest_n_layers[0],
                                                    o.suggest_n_layers[-1])
    dropout = trial.suggest_float(      "dropout",  o.suggest_dropout[0],
                                                    o.suggest_dropout[1], step=0.05)
    lr      = trial.suggest_float(      "lr",       o.suggest_lr[0],
                                                    o.suggest_lr[1], log=True)
    wd      = trial.suggest_float(      "weight_decay", o.suggest_weight_decay[0],
                                                        o.suggest_weight_decay[1], log=True)

    # d_model ต้องหาร n_heads ลงตัว
    if d_model % n_heads != 0:
        raise optuna.exceptions.TrialPruned()

    # ── Build trial model ──────────────────────────────────────
    trial_model_cfg = DDPMTransformerConfig(
        d_model             = d_model,
        ff_mult             = ff_mult,
        num_layers          = n_layers,
        num_attention_heads = n_heads,
        dropout             = dropout,
        timesteps           = m.timesteps,    # ตายตัวจาก exp_cfg
        beta_start          = m.beta_start,
        beta_end            = m.beta_end,
    )

    trial_model = build_model(
        ExperimentConfig(model=trial_model_cfg, training=t),
        meta,
        data,
    )

    trial_optimizer = AdamWConfig(lr=lr, weight_decay=wd).build(trial_model)
    trial_scheduler = t.scheduler.build(trial_optimizer, total_epochs=o.epochs_per_trial)
    trial_criterion = t.criterion.build()

    # ── Mini Engine ────────────────────────────────────────────
    engine = Engine(
        train_loader   = loaders["train"],
        val_loader     = loaders["val"],
        model          = trial_model,
        optimizer      = trial_optimizer,
        criterion      = trial_criterion,
        scheduler      = trial_scheduler,
        max_grad_norm  = t.max_grad_norm,
        clip_gradients = t.use_clip_grad,
        device         = DEVICE,
        checkpoint_dir = str(EXP_DIR / "optuna" / ".tmp"),
    )

    try:
        engine.fit(epochs=o.epochs_per_trial, is_save_best=False, save_every=0,
                   save_plots=False)
        best_val = min(engine.history["val_loss"])
        return best_val
    finally:
        # ── cleanup GPU memory after every trial ──────────────
        engine.history.clear()         # clear history dict before del
        trial_model.cpu()
        del engine
        del trial_model
        del trial_optimizer
        del trial_scheduler
        del trial_criterion

        plt.close("all")
        gc.collect()

        # Sync then flush cache
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    # engine.fit(epochs=o.epochs_per_trial, is_save_best=False, save_every=0)

    # return min(engine.history["val_loss"])

## 4.2 Run study

In [17]:
best_hparams = {}

if not exp_cfg.skip_optuna:
    o = exp_cfg.optuna

    sampler = optuna.samplers.TPESampler(seed=exp_cfg.random_seed)
    pruner  = optuna.pruners.HyperbandPruner(
        min_resource     = o.min_resource,
        max_resource     = o.max_resource,
        reduction_factor = o.reduction_factor,
    )

    study = optuna.create_study(
        direction  = "minimize",
        study_name = EXP_NAME,
        sampler    = sampler,
        pruner     = pruner,
    )

    del model
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    study.optimize(objective, n_trials=o.n_trials, show_progress_bar=True)
    best_hparams = study.best_params

    print(f"\n  Best val loss : {study.best_value:.6f}")
    print(f"  Best params   :\n{json.dumps(best_hparams, indent=4)}")

else:
    print("skip_optuna=True — use params from exp_cfg instead")

skip_optuna=True — use params from exp_cfg instead


## 4.3 Save optuna

In [18]:
if not exp_cfg.skip_optuna:
    optuna_dir = EXP_DIR / "optuna"

    # ── cleanup tmp trial checkpoints ─────────────────────────
    tmp = optuna_dir / ".tmp"
    if tmp.exists():
        shutil.rmtree(tmp)

    # ── study.pkl ──────────────────────────────────────────────
    with open(optuna_dir / "study.pkl", "wb") as f:
        pickle.dump(study, f)

    # ── best_params.json ───────────────────────────────────────
    best_params_out = {
        **best_hparams,
        "dim_feedforward": best_hparams["d_model"] * best_hparams["ff_mult"],
        "best_val_loss"  : study.best_value,
    }
    with open(optuna_dir / "best_params.json", "w") as f:
        json.dump(best_params_out, f, indent=2)

    # ── all_trials.csv ─────────────────────────────────────────
    trials_df = study.trials_dataframe()
    trials_df.to_csv(optuna_dir / "all_trials.csv", index=False)

    # ── HTML plots ─────────────────────────────────────────────
    plots = {
        "optimization_history" : optuna.visualization.plot_optimization_history(study),
        "parallel_coordinate"  : optuna.visualization.plot_parallel_coordinate(study),
        "param_importances"    : optuna.visualization.plot_param_importances(study),
    }
    for name, fig in plots.items():
        pio.write_html(fig, str(optuna_dir / f"{name}.html"))

    print(f"  ✓ study.pkl")
    print(f"  ✓ best_params.json  — best val loss: {study.best_value:.6f}")
    print(f"  ✓ all_trials.csv    — {len(trials_df)} trials")
    print(f"  ✓ optimization_history.html")
    print(f"  ✓ parallel_coordinate.html")
    print(f"  ✓ param_importances.html")
    print(f"\n  → {optuna_dir}")

# 5. Fit Model (Training) 

## 5.1 Build Model (Best param)

In [19]:
if not exp_cfg.skip_training:

    # ── Override model cfg ด้วย best_hparams จาก Optuna ──────
    if best_hparams:
        final_model_cfg = DDPMTransformerConfig(
            d_model             = best_hparams["d_model"],
            ff_mult             = best_hparams["ff_mult"],
            num_layers          = best_hparams["n_layers"],
            num_attention_heads = best_hparams["n_heads"],
            dropout             = best_hparams["dropout"],
            timesteps           = exp_cfg.model.timesteps,
            beta_start          = exp_cfg.model.beta_start,
            beta_end            = exp_cfg.model.beta_end,
        )
        final_optimizer_cfg = AdamWConfig(
            lr           = best_hparams["lr"],
            weight_decay = best_hparams["weight_decay"],
        )
        print("  ✓ Using Optuna best_hparams")
    else:
        # skip_optuna=True → ใช้ exp_cfg ตามที่ตั้งไว้
        final_model_cfg     = exp_cfg.model
        final_optimizer_cfg = exp_cfg.training.optimizer
        print("  ✓ Using exp_cfg defaults (no Optuna)")

    final_exp_cfg = ExperimentConfig(
        **{**asdict(exp_cfg),
           "model"   : final_model_cfg,
           "training": TrainingConfig(
               **{**asdict(exp_cfg.training),
                  "optimizer": final_optimizer_cfg}
           )}
    )

    final_model = build_model(final_exp_cfg, meta, data)
    n_params = sum(p.numel() for p in final_model.parameters() if p.requires_grad)

    print(f"  Model     : {final_model_cfg.name}")
    print(f"  d_model   : {final_model_cfg.d_model}")
    print(f"  n_layers  : {final_model_cfg.num_layers}")
    print(f"  n_heads   : {final_model_cfg.num_attention_heads}")
    print(f"  dropout   : {final_model_cfg.dropout}")
    print(f"  lr        : {final_optimizer_cfg.lr:.2e}")
    print(f"  Params    : {n_params:,}")

  ✓ Using exp_cfg defaults (no Optuna)
  seq_dims   : ('W', 'A')
  feat_dims  : ('D', 'C')
  input_dim  : 80  (20 * 4)
  cond_dim   : 11
  Model     : ddpm_transformer
  d_model   : 1024
  n_layers  : 3
  n_heads   : 4
  dropout   : 0.1
  lr        : 2.60e-04
  Params    : 53,715,024


## 5.2 Train

In [20]:
if not exp_cfg.skip_training:
    t = exp_cfg.training

    optimizer = final_optimizer_cfg.build(final_model)
    scheduler = t.scheduler.build(optimizer, total_epochs=t.num_epochs)
    criterion = t.criterion.build()

    engine = Engine(
        train_loader    = loaders["train"],
        val_loader      = loaders["val"],
        model           = final_model,
        optimizer       = optimizer,
        criterion       = criterion,
        scheduler       = scheduler,
        max_grad_norm   = t.max_grad_norm,
        clip_gradients  = t.use_clip_grad,
        device          = DEVICE,
        checkpoint_dir  = str(EXP_DIR / "checkpoints"),
        seq_dims        = exp_cfg.data.seq_dims,
        feat_dims       = exp_cfg.data.feat_dims,
    )

    engine.fit(
        epochs       = t.num_epochs,
        is_save_best = True,
        save_every   = t.save_checkpoint_freq,
    )

    print(f"\n  ✓ Training complete")
    print(f"  ✓ Plots   → {EXP_DIR / 'plots'}")
    print(f"  ✓ Best    → {EXP_DIR / 'checkpoints' / 'best_model.pt'}")

KeyboardInterrupt: 

In [ ]:
# ── Load best checkpoint ───────────────────────────────────
engine.load_checkpoint("best_model.pt")
print("✓ Best checkpoint loaded")

# ── Quick sanity — single test batch ──────────────────────
x_test, cond_test, init_test, ohlcv_test, date_test, init_date_test = next(iter(loaders["test"]))
x_test    = x_test.to(DEVICE)
cond_test = cond_test.to(DEVICE)
date_test = date_test.to(DEVICE)
init_date_test = init_date_test.to(DEVICE)

with torch.no_grad():
    x_r, cond_r = engine._reshape(x_test, cond_test)
    x_gen       = engine.model.sample(
        x_cond       = cond_r,
        output_shape = x_r.shape,
    )

print(f"  input  shape : {tuple(x_r.shape)}")
print(f"  output shape : {tuple(x_gen.shape)}")
print(f"  output mean  : {x_gen.mean().item():.4f}")
print(f"  output std   : {x_gen.std().item():.4f}")
print(f"  has nan      : {torch.isnan(x_gen).any().item()}")
print(f"  has inf      : {torch.isinf(x_gen).any().item()}")

# ── Shape assertion ────────────────────────────────────────
assert x_gen.shape == x_r.shape, \
    f"Shape mismatch: generated {x_gen.shape} vs input {x_r.shape}"
print("✓ Shape assertion passed")

/home/narodom.y@FUSION.LAB/research/src/engine/trainer.py:330: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=self.device)
/home/narodom.

2026-05-22 19:51:01,006 - Engine - INFO - Loaded checkpoint: /home/narodom.y@FUSION.LAB/research/02_experiments/v1_trial04_ddpm_warmup-cosine_w20_d20_seqWA_featDC_robust_standard/checkpoints/best_model.pt
✓ Best checkpoint loaded
  input  shape : (1, 320, 80)
  output shape : (1, 320, 80)
  output mean  : 0.5417
  output std   : 14.5888
  has nan      : False
  has inf      : False
✓ Shape assertion passed
